<a href="https://colab.research.google.com/github/AtikaDwiAr/Cookpad-Information-Retrieval/blob/main/Project_IR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Anggota Kelompok:
1. Atika Dwi Aryanti (23/511929/PA/21857)
2. Jocelin Marella Ramadhaniska (23/512463/PA/21882)
3. Iffa Hesti Adlik Putri (23/514098/PA/21977)
4. Meilany Dinda Talitha (23/517897/PA/22220)

# Preprocessing

### Install & Import Library

In [ ]:
!apt-get install openjdk-21-jdk -qq
!pip install pyserini
!pip install Sastrawi

import os, json, csv, string

Selecting previously unselected package fonts-dejavu-core.
(Reading database ... 126435 files and directories currently installed.)
Preparing to unpack .../00-fonts-dejavu-core_2.37-2build1_all.deb ...
Unpacking fonts-dejavu-core (2.37-2build1) ...
Selecting previously unselected package fonts-dejavu-extra.
Preparing to unpack .../01-fonts-dejavu-extra_2.37-2build1_all.deb ...
Unpacking fonts-dejavu-extra (2.37-2build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../02-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package libxxf86dga1:amd64.
Preparing to unpack .../03-libxxf86dga1_2%3a1.1.5-0ubuntu3_amd64.deb ...
Unpacking libxxf86dga1:amd64 (2:1.1.5-0ubuntu3) ...
Selecting previously unselected package x11-utils.
Preparing to unpack .../04-x11-utils_7.7+5build2_amd64.deb ...
Unpacking x11-utils (7.7+5build2) ...
Selecting previously unselected package libatk-wrapper-java.
Pre

In [ ]:
import gdown
import pandas as pd
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import nltk

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# Download file dari google drive
file_id = "14Z7V8NTEpoqF7Uhk1jSPyEtHk3iotgLu"
url = f"https://drive.google.com/uc?id={file_id}"
input_csv = "resepbener.csv"
gdown.download(url, input_csv, quiet=False)

# Load csv
df = pd.read_csv(input_csv, sep=',', encoding='utf-8')

Downloading...
From: https://drive.google.com/uc?id=14Z7V8NTEpoqF7Uhk1jSPyEtHk3iotgLu
To: /content/resepbener.csv
100%|██████████| 580k/580k [00:00<00:00, 124MB/s]


### Preprocess

In [ ]:

# Persiapan preprocessing
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stop_words = set(stopwords.words('indonesian'))

# Normalisasi domain resep
normalization_dict = {
    "gula pasir": "gula",
    "gula putih": "gula",
    "santan kelapa": "santan",
    "tepung terigu": "terigu",
    "coklat": "cokelat",
    "sup": "sop",
}

# Daftar satuan yang akan dihapus
satuan = [
    "gr", "kg", "ml", "l",    # berat / volume
    "sdt", "sdm",              # sendok teh / sendok makan
    "bh", "btg", "lbr", "cm", # buah, batang, lembar, cm
    "butir", "siung"
]

# Fungsi preprocessing
def preprocess_item(item):
    if pd.isnull(item):
        return ""
    # Lowercase
    item = item.lower()
    # Tokenisasi
    tokens = word_tokenize(item)
    # Hapus angka & simbol
    tokens = [re.sub(r'[^a-zA-Z]', '', t) for t in tokens]
    tokens = [t for t in tokens if t]  # buang token kosong
    # Stopword removal
    tokens = [t for t in tokens if t not in stop_words]
    # Stemming
    tokens = [stemmer.stem(t) for t in tokens]
    # Normalisasi domain
    tokens = [normalization_dict.get(t, t) for t in tokens]
    # Hapus satuan
    tokens = [t for t in tokens if t not in satuan]
    return " ".join(tokens)

def preprocess_bahan(text):
    if pd.isnull(text):
        return ""
    items = [i.strip() for i in text.split(',')]
    items = [preprocess_item(i) for i in items]
    return ", ".join(items)

def preprocess_langkah(text):
    if pd.isnull(text):
        return ""
    items = [i.strip() for i in text.split(',')]
    items = [preprocess_item(i) for i in items]
    return ". ".join(items)

# Menerapkan preprocessing
df_preprocessed = pd.DataFrame()
df_preprocessed['judul'] = df['judul'].apply(preprocess_item)
df_preprocessed['bahan'] = df['bahan'].apply(preprocess_bahan)
df_preprocessed['langkah'] = df['langkah'].apply(preprocess_langkah)

# Simpan hasil csv
df_preprocessed.to_csv("resep_preprocessed.csv", index=False, sep='\t', encoding='latin1') # Changed encoding to latin1

# Menampilkan 5 baris pertama
print(df_preprocessed.head())

                       judul  \
0    ayam suwir sambal matah   
1  homemade chicken luncheon   
2              luncheon ayam   
3   homemade chicken lucheon   
4           chicken luncheon   

                                               bahan  \
0  paha ayam, bawang putih, geprek, jahe, geprek,...   
1  paha dada ayam fillet, putih telur, bawang put...   
2  paha ayam fillet, putih telur, bawang putih ha...   
3  paha ayam fillet, putih telur, bawang putih, b...   
4  paham ayam fillet, putih telur, bawang putih, ...   

                                             langkah  
0  rebus ayam serta bumbu rebus. masak air susut ...  
1  siap bahan bahan halus bawang putih bawang bom...  
2  siap bahan. ayam segar wajib masuk freezer min...  
3  siap bahan fillet paha ayam segar masuk freeze...  
4  siap bahan. chopper bahan bentuk pasta kental ...  


# Indexing dan Query

### BM25 Baseline

In [ ]:
def csv_to_jsonl(csv_path, json_path):
  with open(csv_path, mode="r", encoding="utf-8") as f_in, \
       open(json_path, mode="w", encoding="utf-8") as f_out:

       reader = csv.DictReader(f_in, delimiter="\t")
       for idx, row in enumerate(reader, start=1):
         # ambil tiap kolom
         judul = row.get("judul", "").strip()
         bahan = row.get("bahan", "").strip()
         langkah = row.get("langkah", "").strip()

         contents = f"Judul: {judul} | Bahan: {bahan} | Langkah: {langkah}"

         # tulis ke jsonl
         f_out.write(json.dumps({"id": "d"+str(idx), "contents": contents}) + "\n")

In [ ]:
os.makedirs("collections/resep_json", exist_ok=True)
csv_path = "resep_preprocessed.csv"
jsonl_path = "collections/resep_json/resep_corpus.jsonl"

csv_to_jsonl(csv_path, jsonl_path)

In [ ]:
!python -m pyserini.index.lucene \
  --collection JsonCollection \
  --input collections/resep_json \
  --index indexes/index_resep \
  --generator DefaultLuceneDocumentGenerator \
  --threads 1 \
  --language uspecified \
  --stemmer none \
  --storePositions --storeDocvectors --storeRaw --storeContents

2025-09-23 14:03:56,896 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:205) - Setting log level to INFO
2025-09-23 14:03:56,900 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:208) - ============ Loading Index Configuration ============
2025-09-23 14:03:56,900 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:209) - AbstractIndexer settings:
2025-09-23 14:03:56,901 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:210) -  + DocumentCollection path: collections/resep_json
2025-09-23 14:03:56,901 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:211) -  + CollectionClass: JsonCollection
2025-09-23 14:03:56,901 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:212) -  + Index path: indexes/index_resep
2025-09-23 14:03:56,902 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:213) -  + Threads: 1
2025-09-23 14:03:56,902 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:214) -  + Optimize (merge segments)? false
Sep 23, 2025 2:03:5

In [ ]:
from pyserini.search.lucene import LuceneSearcher
import csv

searcher_resep = LuceneSearcher('indexes/index_resep')
searcher_resep.set_bm25(k1=0.9, b=0.4)

queries = {
    "ayam goreng": '"ayam goreng"',
    "nasi goreng": '"nasi goreng"',
    "bolu pisang": '"bolu pisang"',
    "sop buah naga": '"sop buah naga"',
    "cokelat kukus": '"cokelat kukus"'
}

print("BM25 Retrieval Results\n")

query_results = {}

for label, q in queries.items():
    hits_resep = searcher_resep.search(q, k=700)
    retrieved = [h.docid for h in hits_resep]
    query_results[label] = retrieved

    # tampilkan contoh 10 hasil teratas
    top10_results = hits_resep[:10]

    print(f"Query: {label}")
    for h in top10_results:
      print(f"  {h.docid}\t| score = {h.score:.4f}")
    print("-" * 50)

    print("Matched docs:")
    for h in top10_results:
        doc_resep = searcher_resep.doc(h.docid)

        resep_raw = doc_resep.contents() if doc_resep else ""

        print(f"ID: {h.docid}")
        print(f"{resep_raw}")
        print()
    print("=" * 60 + "\n")

BM25 Retrieval Results

Query: ayam goreng
  d22	| score = 1.8953
  d18	| score = 1.8412
  d71	| score = 1.8340
  d231	| score = 1.7772
  d91	| score = 1.7357
  d33	| score = 1.7336
  d79	| score = 1.7335
  d269	| score = 1.7129
  d52	| score = 1.7129
  d258	| score = 1.7107
--------------------------------------------------
Matched docs:
ID: d22
Judul: ayam goreng serundeng | Bahan: ekor ayam potong, buah jeruk nipis, kelapa parut, bungkus bumbu ayam ungkep yg halus, air, garam, sedap, buah bawang putih, iris tipis, gula aren, minyak goreng, daun jeruk, daun salam, daun pandan, batang serai, geprek | Langkah: cuci ayam bersih. jeruk nipis diam menit tumis bumbu ayam ungkep halus. tambah bumbu aromatik. tumis harum tambah potong ayam aduk bumbu campur. air ayam rendam tambah garam masak air susut tambah kelapa parut tambah sedap masak bumbu serap dalam kelapa parut. angkat ayam bumbu kelapa parut sisih masuk gula aren dalam bumbu kelapa parut. aduk angkat bumbu kelapa parut. peras air.

### BM25 Phrase Query

In [ ]:
!pip install sentence-transformers
!pip install whoosh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.8/468.8 kB 14.6 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
from whoosh.fields import Schema, TEXT, ID
from whoosh.index import create_in
from whoosh.analysis import StemmingAnalyzer

# Definisikan schema
schema = Schema(
    docid=ID(stored=True, unique=True),
    judul=TEXT(stored=True, analyzer=StemmingAnalyzer()),
    bahan=TEXT(stored=True, analyzer=StemmingAnalyzer()),
    langkah=TEXT(stored=True, analyzer=StemmingAnalyzer())
)

# Buat folder index baru
if not os.path.exists("whoosh_index"):
    os.mkdir("whoosh_index")

ix = create_in("whoosh_index", schema)

# Isi index dari CSV hasil preprocessing
df_preprocessed = pd.read_csv("resep_preprocessed.csv", sep="\t", encoding="latin1")

writer = ix.writer()
for idx, row in df_preprocessed.iterrows():
    # gunakan format d1, d2, ...
    writer.add_document(
        docid=f"d{idx+1}",
        judul=row["judul"],
        bahan=row.get("bahan", ""),
        langkah=row.get("langkah", "")
        )
writer.commit()

print("Whoosh index berhasil dibuat.")

Whoosh index berhasil dibuat.


In [ ]:
from pyserini.search.lucene import LuceneSearcher
from whoosh.qparser import MultifieldParser, syntax
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import pandas as pd

# --- Queries ---
queries = {
    "ayam goreng": "ayam goreng",
    "nasi goreng": "nasi goreng",
    "bolu pisang": "bolu pisang",
    "sop buah naga": "sop buah naga",
    "cokelat kukus": "cokelat kukus"
}

from whoosh.index import open_dir
from whoosh.qparser import QueryParser, syntax

# Buka index
ix = open_dir("whoosh_index")
phrase_results = {}

print("=== Phrase Query BM25 ===\n")
with ix.searcher() as searcher:
    qp = MultifieldParser(["judul", "bahan", "langkah"], schema=ix.schema, group=syntax.OrGroup)

    for label, q in queries.items():
        phrase_q = qp.parse(f'"{q}"')   # force phrase query
        hits = searcher.search(phrase_q, limit=None)  # ambil semua hasil

        retrieved = [h["docid"] for h in hits]
        phrase_results[label] = retrieved

        print(f"Query: {label}")
        for h in hits:
            print(f"  {h['docid']}\t| score = {h.score:.4f} | {h['judul']} | {h['bahan']} | {h['langkah']}")
        print("="*60 + "\n")


=== Phrase Query BM25 ===

Query: ayam goreng
  d22	| score = 14.8009 | ayam goreng serundeng | ekor ayam potong, buah jeruk nipis, kelapa parut, bungkus bumbu ayam ungkep yg halus, air, garam, sedap, buah bawang putih, iris tipis, gula aren, minyak goreng, daun jeruk, daun salam, daun pandan, batang serai, geprek | cuci ayam bersih. jeruk nipis diam menit tumis bumbu ayam ungkep halus. tambah bumbu aromatik. tumis harum tambah potong ayam aduk bumbu campur. air ayam rendam tambah garam masak air susut tambah kelapa parut tambah sedap masak bumbu serap dalam kelapa parut. angkat ayam bumbu kelapa parut sisih masuk gula aren dalam bumbu kelapa parut. aduk angkat bumbu kelapa parut. peras air. goreng bumbu kelapa warna emas tiris bumbu kelapa kering. goreng ayam warna emas goreng bawang putih warna emas angkat tiris taruh potong ayam dalam wadah. tabur kelapa serundeng bawang putih goreng ayam goreng serundeng hidang
  d277	| score = 14.2772 | nasi kuning bumbu ayam goreng instan | beras

# Evaluasi

In [ ]:
# Download file dari google drive
file_id = "1iwZsxDDGBDuqKBqf1BGqKoWtyCcXftY1"
url = f"https://drive.google.com/uc?id={file_id}"
input_csv = "ground_truth.csv"
gdown.download(url, input_csv, quiet=False)

# Load csv
df = pd.read_csv(input_csv, sep=',', encoding='utf-8')

Downloading...
From: https://drive.google.com/uc?id=1iwZsxDDGBDuqKBqf1BGqKoWtyCcXftY1
To: /content/ground_truth.csv
100%|██████████| 803/803 [00:00<00:00, 3.27MB/s]


In [ ]:
ground_truth_loaded = {}
gt_df = pd.read_csv("ground_truth.csv")
for _, row in gt_df.iterrows():
    ground_truth_loaded[row["query"]] = row["relevant_docs"].split(";")

# contoh akses
print(ground_truth_loaded["ayam goreng"])


['d2', 'd6', 'd8', 'd20', 'd22', 'd33', 'd34', 'd35', 'd45', 'd68', 'd71', 'd72', 'd80', 'd82', 'd83', 'd84', 'd86', 'd91', 'd182', 'd205', 'd206', 'd207', 'd210', 'd231', 'd277', 'd282', 'd287', 'd297', 'd299', 'd503', 'd512']


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def evaluate_results(results, ground_truth):
    eval_results = {}

    for q, retrieved in results.items():
        relevant = set(ground_truth.get(q, []))
        retrieved_set = set(retrieved)

        # universe dokumen untuk evaluasi
        all_docs = retrieved_set | relevant
        y_true = [1 if d in relevant else 0 for d in all_docs]
        y_pred = [1 if d in retrieved_set else 0 for d in all_docs]

        if not all_docs:  # kalau tidak ada dokumen
            continue

        # metrik
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        acc = accuracy_score(y_true, y_pred)

        eval_results[q] = {
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "accuracy": acc,
            "retrieved": len(retrieved_set),
            "relevant": len(relevant)
        }

    return eval_results

# --- Evaluasi Keseluruhan ---
def summarize_results(results):
    precisions = [res["precision"] for res in results.values()]
    recalls = [res["recall"] for res in results.values()]
    f1s = [res["f1"] for res in results.values()]
    accs = [res["accuracy"] for res in results.values()]

    summary = {
        "precision": sum(precisions) / len(precisions) if precisions else 0,
        "recall": sum(recalls) / len(recalls) if recalls else 0,
        "f1": sum(f1s) / len(f1s) if f1s else 0,
        "accuracy": sum(accs) / len(accs) if accs else 0,
    }
    return summary


### Evaluasi BM25 Baseline

In [ ]:
# --- Jalankan Evaluasi Basic BM25 Query ---
baseline_eval = evaluate_results(query_results, ground_truth_loaded)

# --- Tampilkan per Query ---
print("\n=== Evaluasi Phrase Query BM25 ===")
for q, res in baseline_eval.items():
    print(f"Query: {q}")
    print(f"  Precision: {res['precision']:.4f}")
    print(f"  Recall   : {res['recall']:.4f}")
    print(f"  F1-score : {res['f1']:.4f}")
    print(f"  Accuracy : {res['accuracy']:.4f}")
    print(f"  Retrieved: {res['retrieved']}, Relevant: {res['relevant']}")
    print("="*50)

# --- Tampilkan Keseluruhan ---
phrase_summary = summarize_results(baseline_eval)
print("\n=== Evaluasi Keseluruhan Phrase Query BM25 ===")
for metric, val in phrase_summary.items():
    print(f"{metric.capitalize()}: {val:.4f}")


=== Evaluasi Phrase Query BM25 ===
Query: ayam goreng
  Precision: 0.0807
  Recall   : 1.0000
  F1-score : 0.1494
  Accuracy : 0.0807
  Retrieved: 384, Relevant: 31
Query: nasi goreng
  Precision: 0.1063
  Recall   : 1.0000
  F1-score : 0.1922
  Accuracy : 0.1063
  Retrieved: 348, Relevant: 37
Query: bolu pisang
  Precision: 0.0943
  Recall   : 1.0000
  F1-score : 0.1724
  Accuracy : 0.0943
  Retrieved: 159, Relevant: 15
Query: sop buah naga
  Precision: 0.1217
  Recall   : 1.0000
  F1-score : 0.2171
  Accuracy : 0.1217
  Retrieved: 345, Relevant: 42
Query: cokelat kukus
  Precision: 0.1329
  Recall   : 0.9545
  F1-score : 0.2333
  Accuracy : 0.1321
  Retrieved: 158, Relevant: 22

=== Evaluasi Keseluruhan Phrase Query BM25 ===
Precision: 0.1072
Recall: 0.9909
F1: 0.1929
Accuracy: 0.1070


### Evaluasi BM25 Phrase Query

In [ ]:
# --- Jalankan Evaluasi Phrase Query ---
phrase_eval = evaluate_results(phrase_results, ground_truth_loaded)

# --- Tampilkan per Query ---
print("\n=== Evaluasi Phrase Query BM25 ===")
for q, res in phrase_eval.items():
    print(f"Query: {q}")
    print(f"  Precision: {res['precision']:.4f}")
    print(f"  Recall   : {res['recall']:.4f}")
    print(f"  F1-score : {res['f1']:.4f}")
    print(f"  Accuracy : {res['accuracy']:.4f}")
    print(f"  Retrieved: {res['retrieved']}, Relevant: {res['relevant']}")
    print("="*50)

# --- Tampilkan Keseluruhan ---
phrase_summary = summarize_results(phrase_eval)
print("\n=== Evaluasi Keseluruhan Phrase Query BM25 ===")
for metric, val in phrase_summary.items():
    print(f"{metric.capitalize()}: {val:.4f}")


=== Evaluasi Phrase Query BM25 ===
Query: ayam goreng
  Precision: 0.9375
  Recall   : 0.4839
  F1-score : 0.6383
  Accuracy : 0.4688
  Retrieved: 16, Relevant: 31
Query: nasi goreng
  Precision: 0.9474
  Recall   : 0.9730
  F1-score : 0.9600
  Accuracy : 0.9231
  Retrieved: 38, Relevant: 37
Query: bolu pisang
  Precision: 1.0000
  Recall   : 1.0000
  F1-score : 1.0000
  Accuracy : 1.0000
  Retrieved: 15, Relevant: 15
Query: sop buah naga
  Precision: 1.0000
  Recall   : 0.1429
  F1-score : 0.2500
  Accuracy : 0.1429
  Retrieved: 6, Relevant: 42
Query: cokelat kukus
  Precision: 1.0000
  Recall   : 0.1818
  F1-score : 0.3077
  Accuracy : 0.1818
  Retrieved: 4, Relevant: 22

=== Evaluasi Keseluruhan Phrase Query BM25 ===
Precision: 0.9770
Recall: 0.5563
F1: 0.6312
Accuracy: 0.5433


### Perbandingan

In [ ]:
# --- Cetak Ringkasan Keseluruhan ---
baseline_summary = summarize_results(baseline_eval)
phrase_summary = summarize_results(phrase_eval)

print("\n=== Perbandingan ===")
print(f"{'Metric':<10} {'Baseline':<10} {'Phrase Query':<10}")
for metric in baseline_summary.keys():
    print(f"{metric:<10} {baseline_summary[metric]:<10.4f} {phrase_summary[metric]:<10.4f}")



=== Perbandingan ===
Metric     Baseline   Phrase Query
precision  0.1072     0.9770    
recall     0.9909     0.5563    
f1         0.1929     0.6312    
accuracy   0.1070     0.5433    
